In [ ]:
#cell 0
!pip install ijson
!pip install tqdm 
!pip install numpy

In [ ]:
#cell 1
import json
import os
import random
from tqdm.notebook import tqdm
import gc
import ijson
import numpy as np
import codecs # Για τον χειρισμό κωδικοποίησης

# --- Παράμετροι Εισόδου/Εξόδου ---
# Βεβαιώσου ότι το όνομα του dataset σου είναι σωστό εδώ
INPUT_JSON_PATH = '/kaggle/input/bioasq-training-v-2016b-txt/allMeSH_limitjournals_2016.json'
OUTPUT_DIR = '/kaggle/working/'

TRAIN_JSONL_PATH = os.path.join(OUTPUT_DIR, 'train_140k.jsonl')
VAL_JSONL_PATH = os.path.join(OUTPUT_DIR, 'val_30k.jsonl')
TEST_JSONL_PATH = os.path.join(OUTPUT_DIR, 'test_30k.jsonl')

# --- Επιθυμητά Μεγέθη Υποσυνόλων ---
NUM_TRAIN_SAMPLES = 140000
NUM_VAL_SAMPLES = 30000
NUM_TEST_SAMPLES = 30000
TOTAL_SAMPLES_TO_EXTRACT = NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES + NUM_TEST_SAMPLES

# --- Random Seed ---
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Οι παράμετροι ορίστηκαν.")
print(f"Αρχείο εισόδου: {INPUT_JSON_PATH}")
print(f"Συνολικά δείγματα προς εξαγωγή: {TOTAL_SAMPLES_TO_EXTRACT}")

In [ ]:
#cell 2
print(f"\nΒήμα 1: Καταμέτρηση συνολικού αριθμού άρθρων στο {INPUT_JSON_PATH}...")
total_articles_in_file = 0
try:
    # Χρησιμοποιούμε την απλή open() με errors='replace' που δούλεψε στη δοκιμή
    with open(INPUT_JSON_PATH, 'r', encoding='utf-8', errors='replace') as f_text_mode:
        articles_parser = ijson.items(f_text_mode, 'articles.item')
        for _ in tqdm(articles_parser, desc="Καταμέτρηση άρθρων"):
            total_articles_in_file += 1
    print(f"Βρέθηκαν συνολικά {total_articles_in_file:,} άρθρα στο αρχείο εισόδου.")

except FileNotFoundError:
    print(f"ΣΦΑΛΜΑ: Το αρχείο {INPUT_JSON_PATH} δεν βρέθηκε.")
    raise
except ijson.JSONError as e:
    print(f"ΣΦΑΛΜΑ ijson κατά την καταμέτρηση: {e}")
    raise
except Exception as e:
    print(f"Προέκυψε σφάλμα κατά την καταμέτρηση: {e}")
    raise

if total_articles_in_file == 0:
    print("ΣΦΑΛΜΑ: Δεν βρέθηκαν άρθρα. Έλεγξε το μονοπάτι 'articles.item' στο ijson ή το αρχείο σου.")
elif total_articles_in_file < TOTAL_SAMPLES_TO_EXTRACT:
    print(f"ΠΡΟΣΟΧΗ: Ο συνολικός αριθμός άρθρων ({total_articles_in_file}) είναι μικρότερος από τον αριθμό που θέλουμε να εξάγουμε ({TOTAL_SAMPLES_TO_EXTRACT}).")
    print("Θα χρησιμοποιηθούν όλα τα διαθέσιμα άρθρα και τα υποσύνολα θα προσαρμοστούν αναλογικά.")
    ratio = total_articles_in_file / TOTAL_SAMPLES_TO_EXTRACT
    NUM_TRAIN_SAMPLES = int(NUM_TRAIN_SAMPLES * ratio)
    NUM_VAL_SAMPLES = int(NUM_VAL_SAMPLES * ratio)
    # Εξασφάλιση ότι το test set παίρνει τα υπόλοιπα, ακόμα κι αν είναι λίγα
    NUM_TEST_SAMPLES = total_articles_in_file - NUM_TRAIN_SAMPLES - NUM_VAL_SAMPLES 
    if NUM_TEST_SAMPLES < 0: NUM_TEST_SAMPLES = 0 # Αποφυγή αρνητικού αριθμού
    TOTAL_SAMPLES_TO_EXTRACT = total_articles_in_file
    print(f"Νέα μεγέθη: Train={NUM_TRAIN_SAMPLES}, Val={NUM_VAL_SAMPLES}, Test={NUM_TEST_SAMPLES}")

In [ ]:
#cell 3
if total_articles_in_file > 0:
    print(f"\nΒήμα 2: Δημιουργία και ανάμειξη τυχαίων δεικτών για {TOTAL_SAMPLES_TO_EXTRACT} άρθρα...")
    all_indices = np.arange(total_articles_in_file)
    np.random.shuffle(all_indices)
    
    # Επιλογή των δεικτών που θα χρησιμοποιηθούν για τα υποσύνολα
    selected_indices = all_indices[:TOTAL_SAMPLES_TO_EXTRACT]
        
    print(f"Επιλέχθηκαν {len(selected_indices)} δείκτες για εξαγωγή.")
    
    # Διαχωρισμός των επιλεγμένων (και ήδη ανακατεμένων) δεικτών σε train, val, test sets
    train_indices_set = set(selected_indices[:NUM_TRAIN_SAMPLES])
    val_indices_set = set(selected_indices[NUM_TRAIN_SAMPLES : NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES])
    test_indices_set = set(selected_indices[NUM_TRAIN_SAMPLES + NUM_VAL_SAMPLES:])
    
    print(f"Δείκτες για Train: {len(train_indices_set)}")
    print(f"Δείκτες για Validation: {len(val_indices_set)}")
    print(f"Δείκτες για Test: {len(test_indices_set)}")
    
    # Έλεγχος για επικαλύψεις (θα πρέπει να είναι 0)
    intersection_train_val = train_indices_set.intersection(val_indices_set)
    intersection_train_test = train_indices_set.intersection(test_indices_set)
    intersection_val_test = val_indices_set.intersection(test_indices_set)
    print(f"Επικάλυψη train-val: {len(intersection_train_val)}")
    print(f"Επικάλυψη train-test: {len(intersection_train_test)}")
    print(f"Επικάλυψη val-test: {len(intersection_val_test)}")

    del all_indices
    del selected_indices
    gc.collect()
else:
    print("Παράλειψη Βήματος 2 καθώς δεν υπάρχουν άρθρα για επεξεργασία.")
    train_indices_set, val_indices_set, test_indices_set = set(), set(), set()

In [ ]:
# Cell 4 (Δοκιμή Ε - Εγγραφή από Λίστες) - ΔΙΟΡΘΩΜΕΝΟ
if total_articles_in_file > 0 and TOTAL_SAMPLES_TO_EXTRACT > 0:
    print("\nΒήμα 3: Ανάγνωση επιλεγμένων άρθρων σε λίστες...")
    
    train_articles_list = []
    val_articles_list = []
    test_articles_list = []
    
    # Μετρητές για να ξέρουμε πόσα άρθρα έχουμε συλλέξει για κάθε split
    # ώστε να σταματήσουμε τη συλλογή όταν φτάσουμε το TOTAL_SAMPLES_TO_EXTRACT
    collected_for_train = 0
    collected_for_val = 0
    collected_for_test = 0
    
    try:
        with open(INPUT_JSON_PATH, 'r', encoding='utf-8', errors='replace') as infile:
            articles_parser = ijson.items(infile, 'articles.item')
            current_article_index = 0
            
            for article_data in tqdm(articles_parser, total=total_articles_in_file, desc="Συλλογή άρθρων"):
                # Αν έχουμε ήδη συλλέξει αρκετά άρθρα συνολικά, μπορούμε να σταματήσουμε νωρίς
                # (αν και οι δείκτες είναι τυχαίοι, οπότε μπορεί να χρειαστεί να διαβάσουμε μεγάλο μέρος του αρχείου)
                if (collected_for_train + collected_for_val + collected_for_test) >= TOTAL_SAMPLES_TO_EXTRACT:
                    # Επιπλέον έλεγχος για να σιγουρευτούμε ότι έχουμε πάρει τον σωστό αριθμό για κάθε split
                    # Αυτό είναι πιο περίπλοκο αν οι δείκτες είναι διάσπαρτοι.
                    # Η πιο απλή προσέγγιση είναι να συνεχίσουμε να διαβάζουμε μέχρι το τέλος του αρχείου
                    # ή μέχρι να βρούμε όλους τους δείκτες που χρειαζόμαστε.
                    # Για τώρα, ας υποθέσουμε ότι το TOTAL_SAMPLES_TO_EXTRACT είναι ο κύριος οδηγός.
                    # Μια καλύτερη συνθήκη break θα ήταν αν ξέραμε τον μέγιστο δείκτη από τα selected_indices.
                    pass # Θα συνεχίσουμε να διαβάζουμε για να πιάσουμε όλους τους επιλεγμένους δείκτες

                filtered_article = {
                    'pmid': article_data.get('pmid', ''),
                    'title': article_data.get('title', ''),
                    'abstractText': article_data.get('abstractText', ''),
                    'meshMajor': article_data.get('meshMajor', []),
                    'year': article_data.get('year', None),
                    'journal': article_data.get('journal', None)
                }
                
                if current_article_index in train_indices_set:
                    if collected_for_train < NUM_TRAIN_SAMPLES:
                        train_articles_list.append(filtered_article)
                        collected_for_train += 1
                elif current_article_index in val_indices_set:
                    if collected_for_val < NUM_VAL_SAMPLES:
                        val_articles_list.append(filtered_article)
                        collected_for_val += 1
                elif current_article_index in test_indices_set:
                    if collected_for_test < NUM_TEST_SAMPLES:
                        test_articles_list.append(filtered_article)
                        collected_for_test += 1
                
                current_article_index += 1

                # Έξοδος αν έχουμε συλλέξει όλα όσα χρειαζόμαστε *και* έχουμε περάσει τον μεγαλύτερο δυνατό δείκτη
                # (Αυτό είναι δύσκολο να το ξέρουμε χωρίς να έχουμε το selected_indices εδώ)
                # Ας βασιστούμε στο ότι το tqdm θα τελειώσει όταν διαβαστεί όλο το αρχείο.
        
        print(f"\nTrain list: Συλλέχθηκαν {len(train_articles_list)} άρθρα (αναμένονταν {NUM_TRAIN_SAMPLES})")
        print(f"Validation list: Συλλέχθηκαν {len(val_articles_list)} άρθρα (αναμένονταν {NUM_VAL_SAMPLES})")
        print(f"Test list: Συλλέχθηκαν {len(test_articles_list)} άρθρα (αναμένονταν {NUM_TEST_SAMPLES})")

        # ΔΙΟΡΘΩΜΕΝΗ ΣΥΝΑΡΤΗΣΗ - Η μόνη αλλαγή είναι το '\n' αντί για '\\n'
        def write_list_to_jsonl(article_list, path, name):
            print(f"Εγγραφή {name} set ({len(article_list)} άρθρα) στο {path}...")
            with open(path, 'w', encoding='utf-8') as outfile:
                for article in tqdm(article_list, desc=f"Saving {name}"):
                    json.dump(article, outfile, ensure_ascii=False)
                    outfile.write('\n')  # ΔΙΟΡΘΩΣΗ: ΕΝΑ backslash αντί για δύο!
            print(f"Η εγγραφή του {name} ολοκληρώθηκε.")

        if train_articles_list: # Έλεγχος αν η λίστα δεν είναι κενή
            write_list_to_jsonl(train_articles_list, TRAIN_JSONL_PATH, "Train")
        else:
            print(f"Η λίστα Train είναι κενή, δεν δημιουργείται το αρχείο {TRAIN_JSONL_PATH}.")
            open(TRAIN_JSONL_PATH, 'w').close() # Δημιουργία κενού αρχείου

        del train_articles_list; gc.collect()
        
        if val_articles_list:
            write_list_to_jsonl(val_articles_list, VAL_JSONL_PATH, "Validation")
        else:
            print(f"Η λίστα Validation είναι κενή, δεν δημιουργείται το αρχείο {VAL_JSONL_PATH}.")
            open(VAL_JSONL_PATH, 'w').close()

        del val_articles_list; gc.collect()
        
        if test_articles_list:
            write_list_to_jsonl(test_articles_list, TEST_JSONL_PATH, "Test")
        else:
            print(f"Η λίστα Test είναι κενή, δεν δημιουργείται το αρχείο {TEST_JSONL_PATH}.")
            open(TEST_JSONL_PATH, 'w').close()
            
        del test_articles_list; gc.collect()

        print("Όλα τα υποσύνολα αποθηκεύτηκαν σε αρχεία .jsonl από τις λίστες.")

    except Exception as e:
        print(f"Προέκυψε σφάλμα κατά την εξαγωγή (Δοκιμή Ε): {e}")
        raise
else:
    print("Παράλειψη Βήματος 3.")
gc.collect()

In [ ]:
#cell 5
def count_lines(filepath):
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return sum(1 for line in f)
    except FileNotFoundError:
        print(f"Το αρχείο {filepath} δεν βρέθηκε για καταμέτρηση γραμμών.")
        return 0

print("\nΈλεγχος αριθμού γραμμών στα αρχεία εξόδου:")
print(f"Γραμμές στο {TRAIN_JSONL_PATH}: {count_lines(TRAIN_JSONL_PATH):,}")
print(f"Γραμμές στο {VAL_JSONL_PATH}: {count_lines(VAL_JSONL_PATH):,}")
print(f"Γραμμές στο {TEST_JSONL_PATH}: {count_lines(TEST_JSONL_PATH):,}")

print("\nΠροεπισκόπηση μερικών γραμμών από το train set (αν υπάρχει):")
try:
    with open(TRAIN_JSONL_PATH, 'r', encoding='utf-8') as f:
        for i in range(3): # Εμφάνιση των πρώτων 3 γραμμών
            line = f.readline().strip()
            if line:
                print(json.loads(line))
            else:
                if i == 0: print("Το αρχείο train είναι κενό.")
                break
except FileNotFoundError:
    print(f"Το αρχείο {TRAIN_JSONL_PATH} δεν βρέθηκε για προεπισκόπηση.")
except json.JSONDecodeError as e:
    print(f"Σφάλμα αποκωδικοποίησης JSON κατά την προεπισκόπηση του {TRAIN_JSONL_PATH}: {e}")
    with open(TRAIN_JSONL_PATH, 'r', encoding='utf-8') as f_err:
        print("Περιεχόμενο πρώτης γραμμής (για debugging):")
        print(f_err.readline()[:500] + "...") # Εκτύπωση των πρώτων 500 χαρακτήρων της πρώτης γραμμής